In [1]:
# imports
from huggingface_hub import snapshot_download
from sklearn.model_selection import train_test_split
import pandas as pd
import os

In [8]:
repo_id = "hu-lab/TFBS_Prediction"
repo_type = "dataset"
local_dir = "../data/raw"

In [ ]:
# download dataset
snapshot_download(repo_id = repo_id, repo_type = repo_type, local_dir = local_dir)

In [23]:
%%bash

# count number of samples per file
local_dir="../data/raw"
for tf in ${local_dir}/*/; do
    tf=${tf%/}
    tf=${tf##*/}
    echo "${tf}"
    for dataset in ${local_dir}/${tf}/*.csv; do
        ans=$(( $(wc -l < ${dataset}) - 1))
        dataset=${dataset%.csv}
        dataset=${dataset##*/}
        echo "${dataset}: ${ans}"
    done
done

ALF2
test: 10835
train: 39003
val: 4334
ALF7
test: 52454
train: 47208
val: 5246
bHLH108
test: 5558
train: 20005
val: 2223
bHLH129
test: 15466
train: 55674
val: 6186
bHLH145
test: 2538
train: 9135
val: 1015
bHLH163
test: 11459
train: 41251
val: 4584
bHLH43
test: 12127
train: 43656
val: 4851
bHLH47
test: 9661
train: 34776
val: 3865
bHLH91
test: 6500
train: 23400
val: 2600
bZIP104
test: 2767
train: 9958
val: 1107
bZIP113
test: 1693
train: 6093
val: 678
bZIP38
test: 2374
train: 8542
val: 950
bZIP40
test: 11052
train: 39783
val: 4421
bZIP9
test: 18142
train: 65311
val: 7257
C3H40
test: 2888
train: 10396
val: 1156
CCHH133
test: 23649
train: 85133
val: 9460
COL13
test: 4672
train: 16815
val: 1869
COL18
test: 6294
train: 22656
val: 2518
COL2
test: 4800
train: 17278
val: 1920
COL3
test: 4642
train: 16711
val: 1857
COL7
test: 14617
train: 52618
val: 5847
COL8
test: 2796
train: 10062
val: 1118
DOF17
test: 14229
train: 51223
val: 5692
DOF21
test: 846
train: 3045
val: 339
DOF22
test: 15507
train: 5

In [33]:
# check sequence length stats per TF
for tf in os.listdir(local_dir):
    if "." in tf:
        continue

    print(tf)
    for dataset in os.listdir(f"{local_dir}/{tf}"):
        df = pd.read_csv(f"{local_dir}/{tf}/{dataset}")
        print(dataset.split('.')[0])
        print(df["sequence"].str.len().agg(["mean", "max", "min"]))

MYB4
train
mean    999.0
max     999.0
min     999.0
Name: sequence, dtype: float64
val
mean    999.0
max     999.0
min     999.0
Name: sequence, dtype: float64
test
mean    999.0
max     999.0
min     999.0
Name: sequence, dtype: float64
OsABF1
train
mean    999.0
max     999.0
min     999.0
Name: sequence, dtype: float64
val
mean    999.0
max     999.0
min     999.0
Name: sequence, dtype: float64
test
mean    999.0
max     999.0
min     999.0
Name: sequence, dtype: float64
bZIP9
train
mean    999.0
max     999.0
min     999.0
Name: sequence, dtype: float64
val
mean    999.0
max     999.0
min     999.0
Name: sequence, dtype: float64
test
mean    999.0
max     999.0
min     999.0
Name: sequence, dtype: float64
OsMYC2
train
mean    999.0
max     999.0
min     999.0
Name: sequence, dtype: float64
val
mean    999.0
max     999.0
min     999.0
Name: sequence, dtype: float64
test
mean    999.0
max     999.0
min     999.0
Name: sequence, dtype: float64
MYBR17
train
mean    999.0
max     999.

In [34]:
%%bash

# merge raw data, add TF (directory) as feature, and remove spaces
# exclude MYB61 [sequences are not 500 bp]
shopt -s extglob
dataset_types=("test" "train" "val")
for type in "${dataset_types[@]}"; do
    echo "tf,sequence,label" > ../data/merged/${type}.csv
    grep -v "sequence" ../data/raw/!(MYB61)/${type}.csv | \
        sed 's/:/,/g' | \
        sed 's#\.\./data/raw/##g' | \
        sed -E 's#/[^,]*##g' | \
        tr -d '[:blank:]' >> ../data/merged/${type}.csv
done

In [4]:
dataset_types = ["test", "train", "val"]
merged_dir = "../data/merged"

In [36]:
# check number of samples and sequence length stats after merging
for dataset in os.listdir(merged_dir):
    df = pd.read_csv(f"{merged_dir}/{dataset}")
    print(dataset.split('.')[0])
    print(df.shape)
    print(df.columns)
    print(df["sequence"].str.len().agg(["mean", "max", "min"]))

train
(2439688, 3)
Index(['tf', 'sequence', 'label'], dtype='str')
mean    500.0
max     500.0
min     500.0
Name: sequence, dtype: float64
val
(271117, 3)
Index(['tf', 'sequence', 'label'], dtype='str')
mean    500.0
max     500.0
min     500.0
Name: sequence, dtype: float64
test
(717089, 3)
Index(['tf', 'sequence', 'label'], dtype='str')
mean    500.0
max     500.0
min     500.0
Name: sequence, dtype: float64


In [37]:
# make stratify key via tf-label concat, split X and y, then remove pre-split files
for type in dataset_types:
    df = pd.read_csv(f"{merged_dir}/{type}.csv")
    df_X = df["sequence"]
    df["stratify_key"] = df["tf"] + '_' + df["label"].astype("string")
    df_y = df["stratify_key"]

    df_X.to_csv(f"{merged_dir}/X_{type}.csv", index = False)
    df_y.to_csv(f"{merged_dir}/y_{type}.csv", index = False)

    os.remove(f"{merged_dir}/{type}.csv")

In [38]:
random_state = 23

In [39]:
# get 10% of test, train, val
for type in dataset_types:
    df_X = df = pd.read_csv(f"{merged_dir}/X_{type}.csv")
    df_y = df = pd.read_csv(f"{merged_dir}/y_{type}.csv")

    small_X, _, small_y, _ = train_test_split(
        df_X,
        df_y,
        train_size = 0.1,
        stratify = df_y,
        random_state = random_state
    )

    small_X.to_csv(f"{merged_dir}/small_X_{type}.csv", index = False)
    small_y.to_csv(f"{merged_dir}/small_y_{type}.csv", index = False)

In [43]:
# check number of samples and sequence length stats after subsampling
for dataset in os.listdir(merged_dir):
    if "small_X" not in dataset:
        continue
    
    df = pd.read_csv(f"{merged_dir}/{dataset}")
    print(dataset.split('.')[0])
    print(df.shape)
    print(df.columns)
    print(df["sequence"].str.len().agg(["mean", "max", "min"]))

small_X_train
(243968, 1)
Index(['sequence'], dtype='str')
mean    500.0
max     500.0
min     500.0
Name: sequence, dtype: float64
small_X_val
(27111, 1)
Index(['sequence'], dtype='str')
mean    500.0
max     500.0
min     500.0
Name: sequence, dtype: float64
small_X_test
(71708, 1)
Index(['sequence'], dtype='str')
mean    500.0
max     500.0
min     500.0
Name: sequence, dtype: float64


In [40]:
# for each dataset, split up stratify key into TF and label
for dataset in os.listdir(merged_dir):
    if "X" in dataset:
        continue
    
    df = pd.read_csv(f"{merged_dir}/{dataset}")
    df[["tf", "label"]] = df["stratify_key"].str.split('_', expand = True)
    df = df.drop(columns = "stratify_key")
    df.to_csv(f"{merged_dir}/{dataset}", index = False)

In [2]:
temp_dir = "../data/temp"

In [42]:
# convert X datasets into FASTA format
for dataset in os.listdir(merged_dir):
    if "y" in dataset:
        continue

    output_path = f"{temp_dir}/{dataset.split('.')[0]}.fasta"
    with open(f"{merged_dir}/{dataset}", "r") as input, open(output_path, "w") as output:
        i = 0
        for line in input:
            if i == 0:
                i += 1
                continue

            output.write(f">seq_{i}\n")
            output.write(line)
            i += 1

In [5]:
%%bash

# for each small_X dataset, generate Fourier Transform features per representation
dataset_types=("test" "train" "val")
for type in "${dataset_types[@]}"; do
    for i in {1..7}; do
        python ../external/MathFeature/methods/FourierClass.py \
            -i ../data/temp/small_X_${type}.fasta \
            -o ../data/temp/small_X_${type}.fourier_${i}.csv \
            -l ${type}_${i} \
            -r ${i} > /dev/null
    done
done


In [8]:
%%bash

# for each small_X dataset, generate CGR Fourier Transform feature
dataset_types=("test" "train" "val")
for type in "${dataset_types[@]}"; do
    printf "../data/temp/small_X_${type}.fasta\n${type}_cgr" |
        python ../external/MathFeature/methods/ChaosGameTheory.py \
        -n 1 \
        -o ../data/temp/small_X_${type}.fourier_cgr.csv \
        -r 3 > /dev/null
done

In [26]:
ft_names = ["binary", "zcurve", "real", "integer", "eiip", "complex", "atomic"]
processed_dir = "../data/processed"

In [27]:
# per small_X, add row_number as feature "nameseq"
# and merge FT features per small_* dataset

for dataset in os.listdir(merged_dir):
    if "small_X" not in dataset:
        continue

    df = pd.read_csv(f"{merged_dir}/{dataset}")
    df["nameseq"] = [f"seq_{i}" for i in range(1, len(df) + 1)]
    print(dataset)

    for ft_set in os.listdir(temp_dir):
        if dataset.split('.')[0] not in ft_set or "fasta" in ft_set:
            continue

        df_ft = pd.read_csv(f"{temp_dir}/{ft_set}")
        new_cols = []
        ft_name = ft_set.split('.')[1].split('_')[1]
        if ft_name != "cgr":
            ft_name = ft_names[int(ft_name) - 1] # more informative

        df_ft = df_ft.drop(columns = "label") # unnecessary
        for old_col in df_ft.columns:
            new_col = old_col
            if old_col != "nameseq":
                new_col = f"{ft_name}_{old_col}"
            
            new_cols.append(new_col)
        
        df_ft.columns = new_cols
        df = pd.merge(df, df_ft, on = "nameseq")
    
    print(df.columns)
    df.to_csv(f"{processed_dir}/{dataset}", index = False)


small_X_train.csv
Index(['sequence', 'nameseq', 'integer_average', 'integer_median',
       'integer_maximum', 'integer_minimum', 'integer_peak',
       'integer_none_levated_peak', 'integer_sample_standard_deviation',
       'integer_population_standard_deviation',
       ...
       'zcurve_percentile25', 'zcurve_percentile50', 'zcurve_percentile75',
       'zcurve_amplitude', 'zcurve_variance', 'zcurve_interquartile_range',
       'zcurve_semi_interquartile_range', 'zcurve_coefficient_of_variation',
       'zcurve_skewness', 'zcurve_kurtosis'],
      dtype='object', length=154)
small_X_val.csv
Index(['sequence', 'nameseq', 'integer_average', 'integer_median',
       'integer_maximum', 'integer_minimum', 'integer_peak',
       'integer_none_levated_peak', 'integer_sample_standard_deviation',
       'integer_population_standard_deviation',
       ...
       'atomic_percentile25', 'atomic_percentile50', 'atomic_percentile75',
       'atomic_amplitude', 'atomic_variance', 'atomic_interqu

In [ ]:
%%bash

# move small_y_* to data/processed/
mv ../data/merged/small_y_* ../data/processed/